In [61]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
import random
from EloTracker import EloTracker

print("Import successful - no need to install ")

Import successful - no need to install 


data prep

In [62]:
#import the data
data = "/Users/dtruongs/Documents/GitHub/world cup 2026 prediction/international_football/results.csv"

df = pd.read_csv(data)

#df.head(10)
df.tail(10)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49368,2026-06-26,Cape Verde,Saudi Arabia,NaN,NaN,FIFA World Cup,Houston,United States,True
49369,2026-06-26,Uruguay,Spain,NaN,NaN,FIFA World Cup,Zapopan,Mexico,True
49370,2026-06-26,Norway,France,NaN,NaN,FIFA World Cup,Foxborough,United States,True
49371,2026-06-26,Senegal,Iraq,NaN,NaN,FIFA World Cup,Toronto,Canada,True
49372,2026-06-27,Algeria,Austria,NaN,NaN,FIFA World Cup,Kansas City,United States,True
49373,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True
49374,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True
49375,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True
49376,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True
49377,2026-06-27,Croatia,Ghana,NaN,NaN,FIFA World Cup,Philadelphia,United States,True


In [63]:
dict_correct = {
    "Cape Verde": "Cabo Verde",
    "Czech Republic":"Czechia",
    "Ireland":"Republic of Ireland",
    "DR Congo": "Congo DR",
    "United States":"USA",
    "Iran": "IR Iran",
    "South Korea":"Korea Republic",
    "Ivory Coast": "Côte d'Ivoire",
    "Turkey": "Türkiye",
    "Gambia": "The Gambia"
}

df['home_team'] = df['home_team'].replace(dict_correct)
df['away_team'] = df['away_team'].replace(dict_correct)

date analyzing

In [64]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
matchs = len(df)
most_old = min(df['date'])
most_recent = max(df['date'])
print(matchs)
print(most_old)
print(most_recent)

49378
1872-11-30 00:00:00
2026-06-27 00:00:00


In [65]:
df['date'] = pd.to_datetime(df['date'])
df_modern = df[df['date'].dt.year >= 2000].sort_values('date').copy()
df_modern.head(10)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
24062,2000-01-04,Egypt,Togo,2.0,1.0,Friendly,Aswan,Egypt,False
24063,2000-01-07,Tunisia,Togo,7.0,0.0,Friendly,Tunis,Tunisia,False
24064,2000-01-08,Trinidad and Tobago,Canada,0.0,0.0,Friendly,Port of Spain,Trinidad and Tobago,False
24065,2000-01-09,Burkina Faso,Gabon,1.0,1.0,Friendly,Ouagadougou,Burkina Faso,False
24066,2000-01-09,Guatemala,Armenia,1.0,1.0,Friendly,Los Angeles,United States,True
24067,2000-01-09,Côte d'Ivoire,Egypt,2.0,0.0,Friendly,Abidjan,Ivory Coast,False
24068,2000-01-09,Mexico,IR Iran,2.0,1.0,Friendly,Oakland,United States,True
24069,2000-01-11,Bermuda,Canada,0.0,2.0,Friendly,Hamilton,Bermuda,False
24070,2000-01-11,Burkina Faso,Cameroon,2.0,2.0,Friendly,Ouagadougou,Burkina Faso,False
24071,2000-01-13,Senegal,Cameroon,0.0,0.0,Friendly,Dakar,Senegal,False


In [66]:
conditions = [
    df_modern['home_score'] > df_modern['away_score'],
    df_modern['home_score'] == df_modern['away_score'],
    df_modern['home_score'] < df_modern['away_score']
]
choices = [1.0, 0.5, 0.0]
df_modern['home_elo_result'] = np.select(conditions, choices, default=np.nan)

In [67]:
#set up elo tracker 
tracker = EloTracker(k_factor=30)
home_elo = []
away_elo = []

for index, row in df_modern.iterrows():
    if pd.isna(row['home_elo_result']):
        #initialize the ratings for new teams
        home_elo.append(tracker.get_rating(row['home_team']))
        away_elo.append(tracker.get_rating(row['away_team']))

    else: 
        pre_match_home, pre_match_away = tracker.update_ratings(row["home_team"], row["away_team"], row["home_elo_result"])
        home_elo.append(pre_match_home)
        away_elo.append(pre_match_away)

In [68]:
#append to the df 
df_modern['home_elo_pre_match'] = home_elo
df_modern['away_elo_pre_match'] = away_elo
df_modern["del_elo"] = df_modern['home_elo_pre_match'] - df_modern['away_elo_pre_match']

print(df_modern.head(10))

            date            home_team away_team  home_score  away_score  \
24062 2000-01-04                Egypt      Togo         2.0         1.0   
24063 2000-01-07              Tunisia      Togo         7.0         0.0   
24064 2000-01-08  Trinidad and Tobago    Canada         0.0         0.0   
24065 2000-01-09         Burkina Faso     Gabon         1.0         1.0   
24066 2000-01-09            Guatemala   Armenia         1.0         1.0   
24067 2000-01-09        Côte d'Ivoire     Egypt         2.0         0.0   
24068 2000-01-09               Mexico   IR Iran         2.0         1.0   
24069 2000-01-11              Bermuda    Canada         0.0         2.0   
24070 2000-01-11         Burkina Faso  Cameroon         2.0         2.0   
24071 2000-01-13              Senegal  Cameroon         0.0         0.0   

      tournament           city              country  neutral  \
24062   Friendly          Aswan                Egypt    False   
24063   Friendly          Tunis             

In [69]:
GROUPS = {
    "A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],
    "B": ["Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["USA", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curacao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cabo Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"]
}

train the model using logistic regression

In [70]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [75]:
conditions = [
    df_modern['home_score'] > df_modern['away_score'],
    df_modern['home_score'] == df_modern['away_score'],
    df_modern['home_score'] < df_modern['away_score']
]
df_modern['target'] = np.select(conditions, [2, 1, 0], default=np.nan)

df_modern['date'] = pd.to_datetime(df_modern['date'])
train_df = df_modern[df_modern['date'].dt.year < 2026].dropna(subset=['target']).copy()
predict_df = df_modern[df_modern['date'].dt.year == 2026].copy()

#init features and target 
features = ["del_elo"]
target = "target"
X_train = train_df[features]
y_train = train_df[target]

#init the model
model = LogisticRegression(solver='lbfgs')
model.fit(X_train, y_train)

#calculate the historical accuracy 
train_preds = model.predict(X_train)
print(f"Historical Accuracy: {accuracy_score(y_train, train_preds):.2%}\n")

#use the predict_df
X_predict = predict_df[features]
prob = model.predict_proba(X_predict)

predict_df['prob_away_win'] = prob[:, 0]
predict_df['prob_draw'] = prob[:, 1]
predict_df['prob_home_win'] = prob[:, 2]

view_cols = ['date', 'home_team', 'away_team', 'prob_home_win', 'prob_draw', 'prob_away_win']
print("Upcoming 2026 Matches with Probabilities:")
print(predict_df[view_cols].round(3))

Historical Accuracy: 62.58%

Upcoming 2026 Matches with Probabilities:
            date     home_team   away_team  prob_home_win  prob_draw  \
49093 2026-01-03       Senegal       Sudan          0.949      0.047   
49094 2026-01-03          Mali     Tunisia          0.380      0.313   
49096 2026-01-04  South Africa    Cameroon          0.372      0.314   
49095 2026-01-04       Morocco    Tanzania          0.974      0.025   
49098 2026-01-05       Nigeria  Mozambique          0.859      0.119   
...          ...           ...         ...            ...        ...   
49374 2026-06-27      Colombia    Portugal          0.426      0.308   
49376 2026-06-27        Panama     England          0.099      0.246   
49372 2026-06-27       Algeria     Austria          0.490      0.294   
49373 2026-06-27        Jordan   Argentina          0.015      0.116   
49377 2026-06-27       Croatia       Ghana          0.902      0.086   

       prob_away_win  
49093          0.004  
49094          0.3

/var/folders/t5/fyh9jh3574j1t7mv4nsl_80w0000gn/T/ipykernel_23169/3816881649.py:36: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(predict_df[view_cols].round(3))
